# EDA Pipeline — NeuroKit2 Highpass Decomposition

Drop-in replacement for the cvxEDA notebook using NeuroKit2's highpass filter method.

| Original (cvxEDA) | This notebook (highpass) |
|---|---|
| `CVX_Phasic` | `HP_Phasic` |
| `CVX_Tonic` | `HP_Tonic` |
| `CVX_Tonic.csv` | `HP_Tonic.csv` |
| `Tonic_Reshaped.csv` | `HP_Tonic_Reshaped.csv` |
| `Final_Result.csv` | `HP_Final_Result.csv` |

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv("MatLab_EDA_For_R.csv")
df.head()

,SUD_003_01_EDA,SUD_003_02_EDA,SUD_003_04_EDA,SUD_003_06_EDA,SUD_003_08_EDA,SUD_003_09_EDA,SUD_003_10_EDA,SUD_003_11_EDA,SUD_003_12_EDA,SUD_003_14_EDA,...,SUD_003_18_EDA,SUD_003_20_EDA,SUD_003_21_EDA,SUD_003_22_EDA,SUD_003_23_EDA,SUD_003_24_EDA,SUD_003_27_EDA,SUD_003_28_EDA,SUD_003_29_EDA,SUD_003_30_EDA
0,0.356146,0.003844,0.017939,0.002563,0.828076,0.017939,0.006406,0.005125,0.003844,8.589259,...,2.178897,0.006406,0.012813,0.089693,0.011532,0.012813,1.347015,0.007688,0.020504,0.016657
1,0.361270,0.005125,0.016657,0.001281,0.825513,0.016657,0.007688,0.007688,0.002563,9.060399,...,2.185304,0.003844,0.011532,0.101225,0.010251,0.014095,1.476429,0.008969,0.021785,0.017939
2,0.354865,0.005125,0.017939,0.001281,0.824232,0.016657,0.006406,0.007688,0.002563,9.067647,...,2.184022,0.007688,0.011532,0.123008,0.008969,0.011532,1.024219,0.007688,0.020504,0.021783
3,0.354865,0.006407,0.015376,0.001281,0.822950,0.016657,0.006406,0.006406,0.002563,9.035616,...,1.333219,0.008969,0.011532,0.106350,0.008969,0.012813,1.576472,0.006407,0.020504,0.024345
4,0.353584,0.005125,0.016657,0.001281,0.820388,0.017939,0.007688,0.007688,0.003844,9.029209,...,2.062097,0.008969,0.012813,0.098662,0.008969,0.014095,1.584160,0.007688,0.023067,0.021783


## 1. Decompose into Phasic and Tonic (NeuroKit2 highpass)

In [ ]:
import numpy as np
from scipy.linalg import toeplitz
from sklearn.linear_model import Lasso
import time

# -----------------------------
# Helper Functions
# -----------------------------

def chop_data(data, num_chops, overlap):
    N_total = data.shape[0]
    step = int(N_total * (1 - overlap) / num_chops)
    segments = []

    for i in range(num_chops):
        start = i * step
        end = start + int(N_total / num_chops)
        if end <= N_total:
            segments.append(data[start:end, :])

    return np.concatenate(segments, axis=1)


def reconstr_data(chopped, original_shape, num_chops, overlap):
    N_total, num_signals = original_shape
    recon = np.zeros(original_shape)
    counts = np.zeros(original_shape)

    seg_len = chopped.shape[0]
    step = int(seg_len * (1 - overlap))

    idx = 0
    for k in range(chopped.shape[1]):
        start = idx
        end = start + seg_len
        if end <= N_total:
            recon[start:end, :] += chopped[:, [k]]
            counts[start:end, :] += 1
        idx += step

    return recon / (counts + 1e-8)


def make_H(N, tau1, tau2):
    t = np.arange(N)
    h = np.exp(-t / tau1) - np.exp(-t / tau2)
    H = toeplitz(h)
    return np.tril(H)


def make_D(N):
    D = -np.eye(N) + np.eye(N, k=1)
    return D[:-1, :]


def my_lasso(A, b, lam):
    model = Lasso(alpha=lam, fit_intercept=False, max_iter=1000)
    model.fit(A, b)
    return model.coef_.T




# -----------------------------
# Main Pipeline
# -----------------------------

def run_pipeline(eda):
    start_total = time.time()

    # -----------------------------
    # Chop Data
    # -----------------------------
    num_chops = 5
    overlap = 0.85

    Y = chop_data(eda, num_chops, overlap)
    N, K = Y.shape
    eda_len = eda.shape[0]

    # -----------------------------
    # Construct H
    # -----------------------------
    tau1, tau2 = 2, 0.75
    H = make_H(N, tau1, tau2)

    # -----------------------------
    # GMS (Placeholder)
    # -----------------------------
    start = time.time()

    lam = 3 / np.sqrt(max(N, K))
    output = gen_matrix_sep_con(Y, H, lam)

    X_gms = reconstr_data(output["S"], eda.shape, num_chops, overlap)
    X_gms[X_gms < 0] = 0

    gms = {
        "X": X_gms,
        "time": time.time() - start
    }

    # -----------------------------
    # Compressed Sensing (No Chop)
    # -----------------------------
    start = time.time()

    D1 = make_D(eda_len)
    H1 = make_H(eda_len, tau1, tau2)

    A1 = np.hstack([D1 @ H1, np.eye(eda_len - 1)])
    b1 = -np.diff(eda, axis=0)

    l_lam = 0.01
    Z1 = my_lasso(A1, b1, l_lam)

    r_X1 = Z1[:eda_len, :]
    r_X1[r_X1 < 0] = 0

    cs = {
        "X": r_X1,
        "time": time.time() - start
    }

    # -----------------------------
    # Compressed Sensing (Chopped)
    # -----------------------------
    start = time.time()

    D = make_D(N)
    A = np.hstack([D @ H, np.eye(N - 1)])
    b = -np.diff(Y, axis=0)

    Z = my_lasso(A, b, l_lam)
    recovered_X = Z[:N, :]

    X_cs2 = reconstr_data(recovered_X, eda.shape, num_chops, overlap)
    X_cs2[X_cs2 < 0] = 0

    csp = {
        "X": X_cs2,
        "time": time.time() - start
    }

    total_time = time.time() - start_total

    return {
        "gms": gms,
        "cs": cs,
        "csp": csp,
        "time_total": total_time
    }
def soft_threshold(X, tau):
    return np.sign(X) * np.maximum(np.abs(X) - tau, 0)


def svt(X, tau):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    s_thresh = np.maximum(s - tau, 0)
    return U @ np.diag(s_thresh) @ Vt


def gen_matrix_sep(M, H, lam, max_iter=100, tol=1e-6, rho=1.0):
    """
    Solve:
        min ||L||_* + lam ||S||_1
        s.t. L + H S = M

    Using ADMM
    """

    N, K = M.shape
    S = np.zeros((H.shape[1], K))
    L = np.zeros_like(M)
    Y = np.zeros_like(M)  # dual variable

    HtH = H.T @ H
    inv_mat = np.linalg.inv(HtH + np.eye(H.shape[1]))

    for it in range(max_iter):

        # --- L update (SVT) ---
        L = svt(M - H @ S + (1/rho) * Y, 1/rho)

        # --- S update (LASSO-like) ---
        temp = H.T @ (M - L + (1/rho)*Y)
        S = inv_mat @ soft_threshold(temp, lam / rho)

        # --- Dual update ---
        residual = M - L - H @ S
        Y = Y + rho * residual

        # --- Convergence ---
        if np.linalg.norm(residual, 'fro') / np.linalg.norm(M, 'fro') < tol:
            break

    return {"L": L, "S": S}


def gen_matrix_sep_con(Y, H, lam):
    """
    Python version of your MATLAB wrapper
    """

    # --- SVD of H ---
    U, s, Vt = np.linalg.svd(H, full_matrices=False)
    rank = np.sum(s > 1e-8)

    Uk = U[:, :rank]
    Vk = Vt[:rank, :].T
    s_inv = 1 / s[:rank]

    # --- Preconditioning ---
    C = Uk @ np.diag(s_inv) @ Uk.T
    CH = Uk @ Vk.T

    # --- Solve transformed problem ---
    out = gen_matrix_sep(C @ Y, CH, lam)

    # --- Recover original variables ---
    S = out["S"]
    L = Y - H @ S

    return {"S": S, "L": L}

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Step 1: Build data matrix
# -----------------------------
eda_matrix = df.values  # shape: (N, K)

# -----------------------------
# Step 2: Chop + construct H
# -----------------------------
num_chops = 5
overlap = 0.85

Y = chop_data(eda_matrix, num_chops, overlap)
N, K = Y.shape

tau1, tau2 = 2, 0.75
H = make_H(N, tau1, tau2)

# -----------------------------
# Step 3: Run GMS decomposition
# -----------------------------
lam = 3 / np.sqrt(max(N, K))

output = gen_matrix_sep_con(Y, H, lam)

# -----------------------------
# Step 4: Reconstruct full signals
# -----------------------------
X_gms = reconstr_data(output["S"], eda_matrix.shape, num_chops, overlap)
X_gms[X_gms < 0] = 0

# -----------------------------
# Step 5: Convert back to DataFrame
# -----------------------------
HP_Phasic = pd.DataFrame(
    X_gms,
    index=df.index,
    columns=[f'Phasic_GMS_{col}' for col in df.columns]
)

HP_Phasic.head()

,Phasic_GMS_SUD_003_01_EDA,Phasic_GMS_SUD_003_02_EDA,Phasic_GMS_SUD_003_04_EDA,Phasic_GMS_SUD_003_06_EDA,Phasic_GMS_SUD_003_08_EDA,Phasic_GMS_SUD_003_09_EDA,Phasic_GMS_SUD_003_10_EDA,Phasic_GMS_SUD_003_11_EDA,Phasic_GMS_SUD_003_12_EDA,Phasic_GMS_SUD_003_14_EDA,...,Phasic_GMS_SUD_003_18_EDA,Phasic_GMS_SUD_003_20_EDA,Phasic_GMS_SUD_003_21_EDA,Phasic_GMS_SUD_003_22_EDA,Phasic_GMS_SUD_003_23_EDA,Phasic_GMS_SUD_003_24_EDA,Phasic_GMS_SUD_003_27_EDA,Phasic_GMS_SUD_003_28_EDA,Phasic_GMS_SUD_003_29_EDA,Phasic_GMS_SUD_003_30_EDA
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,...,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17,9.357788e-17
4,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,...,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17,8.445663e-17


In [ ]:
# -----------------------------
# Step 1: Build data matrix
# -----------------------------
eda_matrix = df.values  # (N, K)

# -----------------------------
# Step 2: Chop + construct H
# -----------------------------
num_chops = 5
overlap = 0.85

Y = chop_data(eda_matrix, num_chops, overlap)
N, K = Y.shape

tau1, tau2 = 2, 0.75
H = make_H(N, tau1, tau2)

# -----------------------------
# Step 3: Run GMS decomposition
# -----------------------------
lam = 3 / np.sqrt(max(N, K))

output = gen_matrix_sep_con(Y, H, lam)

# -----------------------------
# Step 4: Reconstruct tonic (LOW-RANK)
# -----------------------------
X_tonic = reconstr_data(output["L"], eda_matrix.shape, num_chops, overlap)

# (Optional but often helpful)
X_tonic[X_tonic < 0] = 0

# -----------------------------
# Step 5: Convert to DataFrame
# -----------------------------
HP_Tonic = pd.DataFrame(
    X_tonic,
    index=df.index,
    columns=[f'Tonic_GMS_{col}' for col in df.columns]
)

HP_Tonic.head()

,Tonic_GMS_SUD_003_01_EDA,Tonic_GMS_SUD_003_02_EDA,Tonic_GMS_SUD_003_04_EDA,Tonic_GMS_SUD_003_06_EDA,Tonic_GMS_SUD_003_08_EDA,Tonic_GMS_SUD_003_09_EDA,Tonic_GMS_SUD_003_10_EDA,Tonic_GMS_SUD_003_11_EDA,Tonic_GMS_SUD_003_12_EDA,Tonic_GMS_SUD_003_14_EDA,...,Tonic_GMS_SUD_003_18_EDA,Tonic_GMS_SUD_003_20_EDA,Tonic_GMS_SUD_003_21_EDA,Tonic_GMS_SUD_003_22_EDA,Tonic_GMS_SUD_003_23_EDA,Tonic_GMS_SUD_003_24_EDA,Tonic_GMS_SUD_003_27_EDA,Tonic_GMS_SUD_003_28_EDA,Tonic_GMS_SUD_003_29_EDA,Tonic_GMS_SUD_003_30_EDA
0,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,...,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146,0.356146
1,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,...,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270,0.361270
2,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,...,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865
3,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,...,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865,0.354865
4,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,...,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584,0.353584


In [ ]:
HP_Phasic.to_csv('Chen_Phasic.csv', index=False)
HP_Tonic.to_csv('Chen_Tonic.csv',   index=False)

## 2. Assign condition labels and reshape Tonic into 10-second windows

In [ ]:
%load_ext rpy2.ipython


In [ ]:

%%R -i df,HP_Tonic,HP_Phasic -o result2

assign_condition_by_row <- function(row_index) {
  if (row_index >= 1 && row_index <= 239) return("neutral")
  if (row_index >= 240 && row_index <= 539) return("black")
  if (row_index >= 540 && row_index <= 779) return("positive")
  if (row_index >= 780 && row_index <= 1075) return("black")
  if (row_index >= 1076 && row_index <= 1300) return("negative")
  return(NA)
}

HP_Phasic$condition <- sapply(1:nrow(HP_Phasic), assign_condition_by_row)
HP_Phasic <- HP_Phasic[!is.na(HP_Phasic$condition), ]
HP_Phasic$condition <- factor(HP_Phasic$condition,
                              levels = c("neutral", "positive", "negative", "black"))

reshape_by_condition <- function(df_input, window_size = 10, overlap = 0) {

  chunk_size <- window_size * 4
  colnames_data <- colnames(df_input)[colnames(df_input) != "condition"]

  rle_cond <- rle(as.character(df_input$condition))
  block_ends <- cumsum(rle_cond$lengths)
  block_starts <- c(1, head(block_ends, -1) + 1)

  chunk_list <- mapply(function(start, end, cond) {
    row_indices <- start:end
    n_rows <- length(row_indices)
    n_chunks <- ceiling(n_rows / chunk_size)

    padded <- c(row_indices, rep(NA, n_chunks * chunk_size - n_rows))
    chunks <- split(padded, ceiling(seq_along(padded) / chunk_size))

    list(chunks = chunks, condition = cond, n_chunks = n_chunks)
  }, block_starts, block_ends, rle_cond$values, SIMPLIFY = FALSE)

  if (overlap > 0) {
    for (i in seq_len(length(chunk_list) - 1)) {
      current_block <- chunk_list[[i]]
      next_block_start <- block_starts[i + 1]

      overlap_indices <- next_block_start:min(next_block_start + overlap - 1, block_ends[i + 1])

      last_chunk <- current_block$chunks[[current_block$n_chunks]]
      real_indices <- last_chunk[!is.na(last_chunk)]

      combined <- c(real_indices, overlap_indices)

      if (length(combined) <= chunk_size) {
        last_chunk <- c(combined, rep(NA, chunk_size - length(combined)))
      } else {
        last_chunk <- combined
      }

      chunk_list[[i]]$chunks[[current_block$n_chunks]] <- last_chunk
    }
  }

  all_chunks <- do.call(c, lapply(chunk_list, function(x) x[["chunks"]]))
  all_conditions <- unlist(lapply(chunk_list, function(b) rep(b$condition, b$n_chunks)))

  n_cols2 <- length(all_chunks)

  df_reshaped_list2 <- lapply(colnames_data, function(col_name) {
    vals <- df_input[[col_name]]
    mat <- sapply(all_chunks, function(idx) {
      n_real <- sum(!is.na(idx))
      out <- rep(NA, max(chunk_size, length(idx)))
      out[seq_along(idx)[!is.na(idx)]] <- vals[idx[!is.na(idx)]]
      out[seq_len(chunk_size)]
    })
    as.data.frame(mat)
  })

  df_reshaped2 <- do.call(cbind, df_reshaped_list2)

  if (n_cols2 > 0) {
    new_colnames2 <- unlist(lapply(colnames_data, function(col_name) {
      paste0(col_name, "_", seq(window_size, n_cols2 * window_size, by = window_size), "s")
    }))
    colnames(df_reshaped2) <- new_colnames2
  } else {
    message("Warning: n_cols2 is 0, no column renaming will occur.")
  }

  df_transposed2 <- as.data.frame(t(df_reshaped2))
  df_transposed2$condition <- factor(all_conditions, levels = c("neutral", "positive", "negative", "black"))

  chunk_row_indices <- sapply(all_chunks, function(idx) {
    paste(idx[!is.na(idx)], collapse = ", ")
  })
  df_transposed2$source_rows <- chunk_row_indices

  return(df_transposed2)
}

result2 <- reshape_by_condition(HP_Phasic)

In [ ]:
result2.head)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V33,V34,V35,V36,V37,V38,V39,V40,condition,source_rows
Phasic_GMS_SUD_003_01_EDA_10s,0.000000e+00,0.000000e+00,0.000000e+00,9.357788e-17,8.445663e-17,1.354756e-16,1.705976e-17,2.107423e-16,0.000000e+00,0.000000e+00,...,1.840105e-18,2.265001e-16,3.673641e-17,1.053375e-16,5.648484e-17,0.000000e+00,2.791714e-16,0.000000e+00,neutral,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
Phasic_GMS_SUD_003_01_EDA_20s,0.000000e+00,0.000000e+00,0.000000e+00,1.754698e-17,2.931579e-17,0.000000e+00,0.000000e+00,5.002854e-17,1.027786e-17,0.000000e+00,...,2.442409e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.059725e-17,6.848346e-18,1.064435e-17,neutral,"41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52..."
Phasic_GMS_SUD_003_01_EDA_30s,0.000000e+00,0.000000e+00,0.000000e+00,1.882415e-17,2.459580e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.506664e-17,...,2.752537e-17,1.170317e-17,0.000000e+00,1.894046e-17,0.000000e+00,1.207165e-17,0.000000e+00,0.000000e+00,neutral,"81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92..."
Phasic_GMS_SUD_003_01_EDA_40s,2.200103e-17,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.104403e-18,7.598371e-18,0.000000e+00,0.000000e+00,1.042193e-17,...,9.151327e-18,0.000000e+00,0.000000e+00,1.566881e-17,0.000000e+00,2.580207e-18,0.000000e+00,0.000000e+00,neutral,"121, 122, 123, 124, 125, 126, 127, 128, 129, 1..."
Phasic_GMS_SUD_003_01_EDA_50s,0.000000e+00,2.088994e-17,8.299287e-18,0.000000e+00,0.000000e+00,3.991970e-17,0.000000e+00,4.332525e-18,7.875137e-18,2.106553e-17,...,2.550888e-17,8.993258e-18,8.341142e-04,0.000000e+00,2.707471e-17,1.629780e-02,0.000000e+00,0.000000e+00,neutral,"161, 162, 163, 164, 165, 166, 167, 168, 169, 1..."


In [ ]:
result2.to_csv('Chen_Phasic_Reshaped.csv', index=True)

In [ ]:
result.to_csv('Chen_Tonic_Reshaped.csv', index=True)

## 3. Summary statistics per window

In [ ]:
numeric_result = result2.drop(columns=['condition', 'source_rows'])

hp_phasic_desc = pd.DataFrame({
    'Mean_Phasic': numeric_result.mean(axis=1),
    'Max_Phasic':  numeric_result.max(axis=1),
    'Min_Phasic':  numeric_result.min(axis=1),
    'SD_Phasic':   numeric_result.std(axis=1)
})

print(hp_phasic_desc)

                                 Mean_Phasic    Max_Phasic  Min_Phasic  \
Phasic_GMS_SUD_003_01_EDA_10s   9.968624e-04  2.347634e-02         0.0   
Phasic_GMS_SUD_003_01_EDA_20s   1.467785e-17  8.252299e-17         0.0   
Phasic_GMS_SUD_003_01_EDA_30s   8.588256e-05  3.435302e-03         0.0   
Phasic_GMS_SUD_003_01_EDA_40s   5.162261e-18  3.531553e-17         0.0   
Phasic_GMS_SUD_003_01_EDA_50s   2.596105e-03  5.161001e-02         0.0   
...                                      ...           ...         ...   
Phasic_GMS_SUD_003_30_EDA_300s  1.255253e-03  2.602909e-02         0.0   
Phasic_GMS_SUD_003_30_EDA_310s  1.274306e-03  3.125957e-02         0.0   
Phasic_GMS_SUD_003_30_EDA_320s  6.632724e-18  3.489381e-17         0.0   
Phasic_GMS_SUD_003_30_EDA_330s  4.434790e-03  1.317028e-01         0.0   
Phasic_GMS_SUD_003_30_EDA_340s  5.643757e-18  3.355893e-17         0.0   

                                   SD_Phasic  
Phasic_GMS_SUD_003_01_EDA_10s   4.079139e-03  
Phasic_GMS_SUD_00

In [ ]:
numeric_result = result.drop(columns=['condition', 'source_rows'])

hp_tonic_desc = pd.DataFrame({
    'Mean_Tonic': numeric_result.mean(axis=1),
    'Max_Tonic':  numeric_result.max(axis=1),
    'Min_Tonic':  numeric_result.min(axis=1),
    'SD_Tonic':   numeric_result.std(axis=1)
})

print(hp_tonic_desc)

                               Mean_Tonic  Max_Tonic  Min_Tonic  SD_Tonic
Tonic_GMS_SUD_003_01_EDA_10s     0.328972   0.361270   0.168485  0.048363
Tonic_GMS_SUD_003_01_EDA_20s     0.160934   0.169118   0.111457  0.011905
Tonic_GMS_SUD_003_01_EDA_30s     0.109875   0.121965   0.081992  0.008714
Tonic_GMS_SUD_003_01_EDA_40s     0.096427   0.247098   0.078468  0.046564
Tonic_GMS_SUD_003_01_EDA_50s     0.221632   0.289618   0.108806  0.032714
...                                   ...        ...        ...       ...
Tonic_GMS_SUD_003_30_EDA_300s    0.150632   0.233034   0.057697  0.059099
Tonic_GMS_SUD_003_30_EDA_310s    0.033400   0.133652   0.012173  0.033335
Tonic_GMS_SUD_003_30_EDA_320s    0.031924   0.110571   0.017939  0.028248
Tonic_GMS_SUD_003_30_EDA_330s    0.017887   0.305099   0.001973  0.050465
Tonic_GMS_SUD_003_30_EDA_340s    0.005202   0.007688   0.002563  0.001259

[782 rows x 4 columns]


## 4. Merge with raw features

In [ ]:
Final_Dataframe = pd.read_csv("Raw_Features.csv")

hp_tonic_desc_for_merge = hp_tonic_desc.copy()
hp_tonic_desc_for_merge['X'] = hp_tonic_desc_for_merge.index.str.replace('tonic_high_pass_', '', regex=False)
hp_result_merged = pd.merge(Final_Dataframe, hp_tonic_desc_for_merge, on='X', how='left')

In [ ]:
hp_phasic_desc_for_merge = hp_phasic_desc.copy()
hp_phasic_desc_for_merge['X'] = hp_phasic_desc_for_merge.index.str.replace('Phasic_high_pass_', '', regex=False)
hp_result_merged = pd.merge(hp_result_merged, hp_phasic_desc_for_merge, on='X', how='left')

hp_result_merged.head()

,Unnamed: 0,X,V1,V2,V3,V4,V5,V6,V7,V8,...,skewness_raw,mean_raw,Mean_Tonic,Max_Tonic,Min_Tonic,SD_Tonic,Mean_Phasic,Max_Phasic,Min_Phasic,SD_Phasic
0,1,SUD_003_01_EDA_10s,0.356146,0.361270,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,...,-3.042697,0.318193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,SUD_003_01_EDA_20s,0.333086,0.330524,0.331805,0.329243,0.330524,0.329243,0.329243,0.329243,...,-0.078770,0.321844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,SUD_003_01_EDA_30s,0.307464,0.310026,0.308745,0.308745,0.308745,0.307464,0.307464,0.307464,...,-0.193585,0.302211,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,SUD_003_01_EDA_40s,0.294653,0.294653,0.293372,0.293372,0.295934,0.292091,0.292091,0.292091,...,-0.095485,0.287799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,SUD_003_01_EDA_50s,0.279280,0.277999,0.276718,0.275436,0.276718,0.275436,0.275436,0.275436,...,0.792911,0.271977,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
hp_result_merged.to_csv('Chen_Final_Result.csv', index=False)

In [ ]:
hp_tonic_desc_for_merge = hp_tonic_desc.copy()
hp_tonic_desc_for_merge['X'] = hp_tonic_desc_for_merge.index.str.replace(
    'Tonic_GMS_', '', regex=False
)

print(set(Final_Dataframe['X']) - set(hp_tonic_desc_for_merge['X']))

set()


In [ ]:
# Check how many keys actually match
matches = Final_Dataframe['X'].isin(hp_tonic_desc_for_merge['X']).sum()
print(f"Number of matches found: {matches} out of {len(Final_Dataframe)}")

Number of matches found: 0 out of 782


In [ ]:
# Look at the rows that failed to merge
failed_rows = hp_result_merged[hp_result_merged['some_column_from_hp_tonic'].isna()]
print(failed_rows[['X']].head())

KeyError: 'some_column_from_hp_tonic'

In [ ]:
# Check the first 10 rows where the Tonic data is missing
missing_tonic = hp_result_merged[hp_result_merged['Mean_Tonic'].isna()]

print("--- Samples of 'X' that failed to match ---")
print(missing_tonic['X'].head(10).tolist())

# Check what the keys look like in your source dataframe
print("\n--- Samples of 'X' in the source (hp_tonic_desc_for_merge) ---")
print(hp_tonic_desc_for_merge['X'].head(10).tolist())

--- Samples of 'X' that failed to match ---
['SUD_003_01_EDA_10s', 'SUD_003_01_EDA_20s', 'SUD_003_01_EDA_30s', 'SUD_003_01_EDA_40s', 'SUD_003_01_EDA_50s', 'SUD_003_01_EDA_60s', 'SUD_003_01_EDA_70s', 'SUD_003_01_EDA_80s', 'SUD_003_01_EDA_90s', 'SUD_003_01_EDA_100s']

--- Samples of 'X' in the source (hp_tonic_desc_for_merge) ---
['Tonic_GMS_SUD_003_01_EDA_10s', 'Tonic_GMS_SUD_003_01_EDA_20s', 'Tonic_GMS_SUD_003_01_EDA_30s', 'Tonic_GMS_SUD_003_01_EDA_40s', 'Tonic_GMS_SUD_003_01_EDA_50s', 'Tonic_GMS_SUD_003_01_EDA_60s', 'Tonic_GMS_SUD_003_01_EDA_70s', 'Tonic_GMS_SUD_003_01_EDA_80s', 'Tonic_GMS_SUD_003_01_EDA_90s', 'Tonic_GMS_SUD_003_01_EDA_100s']


In [ ]:
# 1. Fix the Tonic merge keys
hp_tonic_desc_for_merge = hp_tonic_desc.copy()
# Note: Use 'Tonic_GMS_' to match your sample data
hp_tonic_desc_for_merge['X'] = hp_tonic_desc_for_merge.index.str.replace('Tonic_GMS_', '', regex=False)

hp_result_merged = pd.merge(Final_Dataframe, hp_tonic_desc_for_merge, on='X', how='left')

# 2. Fix the Phasic merge keys
# (Double-check if this uses 'Phasic_GMS_' or something similar!)
hp_phasic_desc_for_merge = hp_phasic_desc.copy()
hp_phasic_desc_for_merge['X'] = hp_phasic_desc_for_merge.index.str.replace('Phasic_GMS_', '', regex=False)

hp_result_merged = pd.merge(hp_result_merged, hp_phasic_desc_for_merge, on='X', how='left')

In [ ]:
hp_result_merged.head()

,Unnamed: 0,X,V1,V2,V3,V4,V5,V6,V7,V8,...,skewness_raw,mean_raw,Mean_Tonic,Max_Tonic,Min_Tonic,SD_Tonic,Mean_Phasic,Max_Phasic,Min_Phasic,SD_Phasic
0,1,SUD_003_01_EDA_10s,0.356146,0.361270,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,...,-3.042697,0.318193,0.328972,0.361270,0.168485,0.048363,9.968624e-04,2.347634e-02,0.0,4.079139e-03
1,2,SUD_003_01_EDA_20s,0.333086,0.330524,0.331805,0.329243,0.330524,0.329243,0.329243,0.329243,...,-0.078770,0.321844,0.160934,0.169118,0.111457,0.011905,1.467785e-17,8.252299e-17,0.0,2.233096e-17
2,3,SUD_003_01_EDA_30s,0.307464,0.310026,0.308745,0.308745,0.308745,0.307464,0.307464,0.307464,...,-0.193585,0.302211,0.109875,0.121965,0.081992,0.008714,8.588256e-05,3.435302e-03,0.0,5.431690e-04
3,4,SUD_003_01_EDA_40s,0.294653,0.294653,0.293372,0.293372,0.295934,0.292091,0.292091,0.292091,...,-0.095485,0.287799,0.096427,0.247098,0.078468,0.046564,5.162261e-18,3.531553e-17,0.0,8.779513e-18
4,5,SUD_003_01_EDA_50s,0.279280,0.277999,0.276718,0.275436,0.276718,0.275436,0.275436,0.275436,...,0.792911,0.271977,0.221632,0.289618,0.108806,0.032714,2.596105e-03,5.161001e-02,0.0,8.982577e-03
